# Reading Lab — baseline A/B experiments

Uses the same code and prompts as the localhost UI. **No training yet.** Run Sarvam-1 and an OpenAI API model, then score the actual outputs locally. Start with two development cases. Test cases require an explicit freeze switch. API costs are separate from free GPU availability.

Choose **Runtime → Change runtime type → GPU**. Upload `reading-lab-stage1.zip` in the next cell. No book PDFs or credentials are in the zip. This notebook has not yet been executed on a GPU.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io
uploaded = files.upload()
archive = next((v for k,v in uploaded.items() if k.endswith('.zip')), None)
assert archive is not None, 'Upload reading-lab-stage1.zip'
ROOT = Path('/content/reading-lab-stage1')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    for name in z.namelist():
        target = (ROOT / name).resolve()
        assert target.is_relative_to(ROOT), 'Invalid archive path'
    z.extractall(ROOT)
assert (ROOT / 'run_baseline.py').exists()
print('Evaluation kit ready.')

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements-model.txt'), 'bitsandbytes'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun.'
print(torch.cuda.get_device_name(0))

## Persistent results and configuration

Mount your Drive to save each response immediately. Start with `LIMIT=2`. Use a new `RUN_NAME` whenever configuration changes. Keep the same name only to resume an interrupted, unchanged run. The API model must be one your account can access. Review its price before running. The notebook does not pick a paid model for you.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SPLIT = 'dev'
LIMIT = 2
FROZEN = False
RUN_NAME = 'smoke-01'
OPENAI_MODEL = 'gpt-6-luna'  # Enter your accessible API model ID (snapshot ID if available).
OPENAI_OPTIONS = {'reasoning': {'effort': 'none'}}  # e.g. temperature or reasoning only if supported by this model.
OPENAI_MAX_OUTPUT_TOKENS = 512  # Includes reasoning tokens for reasoning models.
OUT = Path('/content/drive/MyDrive/Reading-Model/runs') / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
assert SPLIT in ('dev', 'test')
assert SPLIT != 'test' or (FROZEN and LIMIT == 0), 'Freeze first; use all 50 test cases.'
print('Outputs:', OUT)
print('API calls planned:', 2 * (LIMIT or (20 if SPLIT == 'dev' else 50)))

In [ ]:
import os, getpass, json
assert OPENAI_MODEL.strip(), 'Set OPENAI_MODEL above before starting.'
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (not saved in notebook): ')
common = ['--split', SPLIT, '--limit', str(LIMIT), '--out', str(OUT)]
if FROZEN:
    common += ['--frozen']

## Generate the four conditions

Sarvam uses a completion transcript and greedy generation; OpenAI uses native chat messages with the same instruction text. Model loading/warm-up is excluded from latency. Errors are saved, not hidden. Reruns skip saved records; use a new folder after changing settings. The jobs below make real API requests and download Sarvam weights.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'run_baseline.py'), '--backend', 'sarvam', *common], check=True)

In [ ]:
try:
    subprocess.run([sys.executable, str(ROOT/'run_baseline.py'), '--backend', 'openai', '--model', OPENAI_MODEL,
                    '--options', json.dumps(OPENAI_OPTIONS), '--max-output-tokens', str(OPENAI_MAX_OUTPUT_TOKENS), *common], check=True)
finally:
    os.environ.pop('OPENAI_API_KEY', None)

## Export and review

This reports execution counts only, not quality scores. Download the two result files and import them into the local UI's **Blind review** tab. Rate outputs before revealing identities. Read `RUBRIC.md` and `PROTOCOL.md`. If a job failed, fix it on dev and rerun with `--retry-errors` explicitly; do not silently drop failures from the report.

In [ ]:
from collections import Counter
for path in sorted(OUT.glob(f'*-{SPLIT}.jsonl')):
    latest = {}
    for line in path.read_text().splitlines():
        row = json.loads(line)
        latest[(row['case_id'], row['condition'])] = row
    print(path.name, dict(Counter(r['status'] for r in latest.values())))
    files.download(str(path))

## Optional Claude A/B baseline
Run only when you want the two extra Claude conditions. Uses the same prompts and cases; adds 2 API calls per selected case.

In [ ]:
CLAUDE_MODEL = 'claude-haiku-5-5'  # Enter an accessible Claude model ID.
if CLAUDE_MODEL:
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic key (not saved): ')
    try:
        subprocess.run([sys.executable, str(ROOT/'run_baseline.py'), '--backend', 'claude', '--model', CLAUDE_MODEL, *common], check=True)
        files.download(str(OUT / f'claude-{SPLIT}.jsonl'))
    finally:
        os.environ.pop('ANTHROPIC_API_KEY', None)
